# Initialization

In [12]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [13]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [14]:
items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [15]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01")

train_test_global_time_split_idx = (
    events["started_at"] < train_test_global_time_split_date
)

events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# количество пользователей, которые есть и в train, и в test
common_users = users_train[users_train.isin(users_test)]

print(len(users_train), len(users_test), len(common_users))

536293 138148 131924


In [16]:
cold_users = users_test[~users_test.isin(users_train)]

print(len(cold_users))

6224


In [17]:
from sklearn.preprocessing import MinMaxScaler

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["book_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)

# Сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values(
    by="popularity_score",
    ascending=False
)

# Выбираем топ-100 книг с рейтингом не ниже 4
top_k_pop_items = item_popularity[
    item_popularity["avg_rating"] >= 4
].head(100)

In [18]:
# добавляем информацию о книгах
top_k_pop_items = top_k_pop_items.merge(
    items.set_index("book_id")[["author", "title", "genre_and_votes", "publication_year"]], on="book_id")

with pd.option_context('display.max_rows', 100):
    display(top_k_pop_items[["book_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]])

,book_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,21647,4.069663,0.397052,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,17819,4.065043,0.326464,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,16634,4.331249,0.324709,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,14571,4.163681,0.273431,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
4,17927395,Sarah J. Maas,A Court of Mist and Fury (A Court of Thorns an...,2016,13026,4.443191,0.260846,"{'Fantasy': 10186, 'Romance': 3346, 'Young Adu..."
5,5,"J.K. Rowling, Mary GrandPré",Harry Potter and the Prisoner of Azkaban (Harr...,2004,13231,4.303907,0.256646,"{'Fantasy': 49784, 'Young Adult': 15393, 'Fict..."
6,21853621,Kristin Hannah,The Nightingale,2015,13011,4.313197,0.252923,"{'Historical-Historical Fiction': 10105, 'Fict..."
7,13206900,Marissa Meyer,"Winter (The Lunar Chronicles, #4)",2015,13230,4.220937,0.251679,"{'Fantasy': 4835, 'Young Adult': 4672, 'Scienc..."
8,9969571,Ernest Cline,Ready Player One,2011,12679,4.041801,0.230960,"{'Science Fiction': 10560, 'Fiction': 7713, 'Y..."
9,6,"J.K. Rowling, Mary GrandPré",Harry Potter and the Goblet of Fire (Harry Pot...,2002,11953,4.274408,0.230265,"{'Fantasy': 48257, 'Young Adult': 15483, 'Fict..."


In [19]:
cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(
        top_k_pop_items[["book_id", "avg_rating"]],
        on="book_id",
        how="left"
    )

cold_user_items_no_avg_rating_idx = (
    cold_users_events_with_recs["avg_rating"].isnull()
)

cold_user_recs = cold_users_events_with_recs[
    ~cold_user_items_no_avg_rating_idx
][["user_id", "book_id", "rating", "avg_rating"]]

In [20]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Рассчитываем RMSE
rmse = np.sqrt(
    mean_squared_error(
        cold_user_recs["rating"],
        cold_user_recs["avg_rating"]
    )
)

# Рассчитываем MAE
mae = mean_absolute_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"]
)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

RMSE: 1.65
MAE: 1.12


In [21]:
# посчитаем покрытие холодных пользователей рекомендациями
cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}")

Доля пользователей без релевантных рекомендаций: 0.83
Среднее покрытие пользователей: 0.57


# === Знакомство: первые персональные рекомендации

In [22]:
from surprise import Dataset, Reader
from surprise import SVD

# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'book_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set)

In [23]:
surprise_test_set = list(events_test[['user_id', 'book_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set)

In [24]:
from surprise import accuracy

rmse = accuracy.rmse(svd_predictions)
mae = accuracy.mae(svd_predictions)
                     
print(rmse, mae)

RMSE: 1.3469
MAE:  0.9899
1.3468989130561508 0.9898677202113105


In [25]:
from surprise import NormalPredictor, SVD, accuracy

# Фиксируем генератор случайных чисел
np.random.seed(0)

# 1. Случайная модель NormalPredictor
random_model = NormalPredictor()
random_model.fit(surprise_train_set)

random_predictions = random_model.test(surprise_test_set)

# Рассчитываем MAE случайной модели
random_mae = accuracy.mae(random_predictions, verbose=False)

# 2. Модель SVD
svd_model = SVD(random_state=0)
svd_model.fit(surprise_train_set)

svd_predictions = svd_model.test(surprise_test_set)

# Рассчитываем MAE модели SVD
svd_mae = accuracy.mae(svd_predictions, verbose=False)

# 3. Процент превышения MAE
mae_difference_percent = (random_mae - svd_mae) / svd_mae * 100

print(f"MAE NormalPredictor: {random_mae:.4f}")
print(f"MAE SVD: {svd_mae:.4f}")
print(f"MAE случайной модели выше на: {round(mae_difference_percent)}%")

MAE NormalPredictor: 1.4945
MAE SVD: 0.9899
MAE случайной модели выше на: 51%


# === Базовые подходы: коллаборативная фильтрация

# === Базовые подходы: контентные рекомендации

# === Базовые подходы: валидация

# === Двухстадийный подход: метрики

# === Двухстадийный подход: модель

# === Двухстадийный подход: построение признаков